In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Training Dynamics — Section 6.4.2

Train a small neural network on a toy dataset and compare how different **activation functions** affect convergence speed and final accuracy.

ReLU is empirically shown to converge up to **six times faster** than tanh on datasets like CIFAR-10. This demo lets you observe that effect directly by training a two-layer MLP on circles, XOR, or moons data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


# ── Activations ────────────────────────────────────────────────────────────────
def sigmoid(x):    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))
def d_sigmoid(x):  s = sigmoid(x); return s * (1 - s)
def tanh_fn(x):    return np.tanh(x)
def d_tanh(x):     return 1 - np.tanh(x) ** 2
def softplus(x):   return np.log1p(np.exp(np.clip(x, -30, 30)))
def d_softplus(x): return sigmoid(x)
def relu(x):       return np.maximum(0, x)
def d_relu(x):     return (x > 0).astype(float)
def hinge(x):      return np.maximum(0, x + 1)
def d_hinge(x):    return (x > -1).astype(float)

ACTS = {
    'Sigmoid':  (sigmoid,  d_sigmoid,  '#2563eb'),
    'Tanh':     (tanh_fn,  d_tanh,     '#059669'),
    'Softplus': (softplus, d_softplus, '#d97706'),
    'ReLU':     (relu,     d_relu,     '#dc2626'),
    'Hinge':    (hinge,    d_hinge,    '#7c3aed'),
}


# ── Dataset generators ─────────────────────────────────────────────────────────
def make_circles(n=100, noise=0.1, seed=42):
    rng = np.random.default_rng(seed)
    angles = np.linspace(0, 2 * np.pi, n // 2, endpoint=False)
    r_inner, r_outer = 0.4, 0.9
    X0 = np.c_[r_inner * np.cos(angles), r_inner * np.sin(angles)]
    X1 = np.c_[r_outer * np.cos(angles), r_outer * np.sin(angles)]
    X = np.vstack([X0, X1]) + rng.normal(0, noise, (n, 2))
    y = np.array([0] * (n // 2) + [1] * (n // 2))
    return X, y


def make_xor(n=100, noise=0.1, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, (n, 2))
    y = ((X[:, 0] > 0) != (X[:, 1] > 0)).astype(int)
    X += rng.normal(0, noise, X.shape)
    return X, y


def make_moons(n=100, noise=0.1, seed=42):
    rng = np.random.default_rng(seed)
    half = n // 2
    t = np.linspace(0, np.pi, half)
    X0 = np.c_[np.cos(t), np.sin(t)]
    X1 = np.c_[1 - np.cos(t), -np.sin(t) + 0.5]
    X = np.vstack([X0, X1]) + rng.normal(0, noise, (n, 2))
    y = np.array([0] * half + [1] * half)
    return X, y


# ── MLP ────────────────────────────────────────────────────────────────────────
class TwoLayerMLP:
    def __init__(self, H=8, seed=1):
        rng = np.random.default_rng(seed)
        scale = np.sqrt(2)
        self.W1 = rng.standard_normal((H, 2)) * scale
        self.b1 = np.zeros(H)
        self.W2 = rng.standard_normal((1, H)) * scale / np.sqrt(H)
        self.b2 = np.zeros(1)

    def forward(self, X, act_fn):
        self.X = X
        self.z1 = X @ self.W1.T + self.b1
        self.a1 = act_fn(self.z1)
        self.z2 = self.a1 @ self.W2.T + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2.squeeze()

    def loss(self, y):
        p = np.clip(self.a2.squeeze(), 1e-15, 1 - 1e-15)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def backward(self, y, lr, d_act_fn):
        N = len(y)
        dz2 = (self.a2.squeeze() - y)[:, None] / N
        dW2 = dz2.T @ self.a1
        db2 = dz2.sum(0)
        da1 = dz2 @ self.W2
        dz1 = da1 * d_act_fn(self.z1)
        dW1 = dz1.T @ self.X
        db1 = dz1.sum(0)
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2


def train(act_name='ReLU', dataset='circles', H=8, lr=0.1, steps=500, seed=42):
    fn, dfn, color = ACTS[act_name]

    if dataset == 'circles':
        X, y = make_circles(seed=seed)
    elif dataset == 'xor':
        X, y = make_xor(seed=seed)
    else:
        X, y = make_moons(seed=seed)

    net = TwoLayerMLP(H=H, seed=1)
    losses = []
    for _ in range(steps):
        net.forward(X, fn)
        losses.append(net.loss(y))
        net.backward(y, lr, dfn)

    # Final accuracy
    preds = (net.forward(X, fn) > 0.5).astype(int)
    acc = np.mean(preds == y)

    # Decision boundary grid
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 50), np.linspace(-1.5, 1.5, 50))
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = net.forward(grid, fn).reshape(xx.shape)

    return losses, acc, X, y, xx, yy, probs, color


def draw_training_dynamics(act_name='ReLU', dataset='circles', H=8, lr=0.1, steps=500, seed=42):
    losses, acc, X, y, xx, yy, probs, color = train(act_name, dataset, H, lr, steps, seed)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Loss curve
    axes[0].plot(losses, color=color, lw=2)
    axes[0].set_xlabel('Training step')
    axes[0].set_ylabel('Binary cross-entropy')
    axes[0].set_title(f'Training Loss — {act_name}  (LR={lr})', fontsize=11)
    axes[0].grid(True, alpha=0.3)

    # Decision boundary
    axes[1].contourf(xx, yy, probs, levels=50, cmap='RdBu_r', alpha=0.4)
    axes[1].contour(xx, yy, probs, levels=[0.5], colors='k', linewidths=1)
    mask0 = y == 0
    axes[1].scatter(X[mask0, 0], X[mask0, 1], c='#2563eb', s=30, edgecolors='w', lw=0.5, label='Class 0')
    axes[1].scatter(X[~mask0, 0], X[~mask0, 1], c='#dc2626', s=30, marker='s', edgecolors='w', lw=0.5, label='Class 1')
    axes[1].set_xlim(-1.5, 1.5); axes[1].set_ylim(-1.5, 1.5)
    axes[1].set_xticks([]); axes[1].set_yticks([])
    axes[1].set_title(f'Decision boundary  (Accuracy = {acc:.1%})', fontsize=11)
    axes[1].legend(loc='upper right', fontsize=9)

    plt.suptitle(f'Activation: {act_name}   Dataset: {dataset}   Hidden units: {H}   Steps: {steps}',
                 fontsize=10, y=0.0)
    plt.tight_layout()
    plt.show()


widgets.interact(
    draw_training_dynamics,
    act_name=widgets.Dropdown(
        options=['Sigmoid', 'Tanh', 'Softplus', 'ReLU', 'Hinge'],
        value='ReLU',
        description='Activation',
    ),
    dataset=widgets.Dropdown(
        options=['circles', 'xor', 'moons'],
        value='circles',
        description='Dataset',
    ),
    H=widgets.IntSlider(min=4, max=16, step=2, value=8, description='Hidden units', continuous_update=False),
    lr=widgets.SelectionSlider(
        options=[0.001, 0.01, 0.05, 0.1, 0.2, 0.5],
        value=0.1,
        description='Learning rate',
        continuous_update=False,
    ),
    steps=widgets.IntSlider(min=100, max=2000, step=100, value=500, description='Steps', continuous_update=False),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=42, description='Seed', continuous_update=False),
)